# 多 PDF 按页码分组拼图

功能：将多个 PDF 中的页面，**按相同页码归为一组**，每一组拼成一张 PNG。

例子：输入 3 个 PDF（各 5 页）→ 输出 5 张 PNG
- `page_000.png` = PDF1 第1页 + PDF2 第1页 + PDF3 第1页
- `page_001.png` = PDF1 第2页 + PDF2 第2页 + PDF3 第2页
- ...
- `page_004.png` = PDF1 第5页 + PDF2 第5页 + PDF3 第5页

### 依赖安装
如果缺少依赖，取消注释下面一行安装。

In [19]:
# !pip install pypdf pymupdf pillow

### import

In [20]:
import sys
from pathlib import Path
import math

import fitz  # PyMuPDF
from PIL import Image

# 复用已有的 collage 工具函数
sys.path.append('/home/wuct/ALICE/reps/cfAnRes/tools/template')
from merge_pdf import collage_png_pages_to_single

### 配置

在这里填入你的 PDF 文件路径列表。

In [21]:
# ─────────── 修改这里 ───────────
PDF_PATHS = [
    "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76/raw_yields_00.pdf",
    "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76/raw_yields_07.pdf",
    "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76/raw_yields_14.pdf",
]

OUTPUT_DIR = "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76"   # 输出目录

# 拼图参数
DPI       = 400      # 从 PDF 提取页面的分辨率
PAGE_W    = 1620     # 拼图画布宽（像素）
PAGE_H    = 540     # 拼图画布高（像素）
COLS      = None     # 拼图列数；None = 自动按 16:9 估算
MARGIN    = 1.0      # 每个小图四周留白（像素）
KEEP_TEMP = False    # 是否保留中间临时 PNG（用于调试）
# ────────────────────────────────

### 核心逻辑

1. 用 PyMuPDF 将每个 PDF 的所有页面渲染为 PIL Image
2. 按页码分组：`groups[page_idx] = [img_from_pdf1, img_from_pdf2, ...]`
3. 将每组图片临时写到磁盘
4. 调用 `collage_png_pages_to_single` 拼合成一张 PNG
5. 清理临时文件

In [22]:
def extract_all_pages(pdf_paths, dpi=200):
    """
    从多个 PDF 中提取所有页面，返回:
        all_pages: list[list[PIL.Image]] — all_pages[pdf_idx][page_idx]
        max_pages: int — 所有 PDF 中的最大页数
    """
    all_pages = []
    max_pages = 0
    for pdf_path in pdf_paths:
        p = Path(pdf_path)
        if not p.exists():
            raise FileNotFoundError(f"找不到 PDF: {p}")
        print(f"  正在读取: {p.name}")
        doc = fitz.open(str(p))
        pages = []
        for i, page in enumerate(doc):
            mat = fitz.Matrix(dpi / 72, dpi / 72)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pages.append(img)
        all_pages.append(pages)
        max_pages = max(max_pages, len(pages))
        doc.close()
        print(f"    → {len(pages)} 页")
    return all_pages, max_pages


def group_and_collage(all_pages, max_pages, output_dir,
                      page_w=1920, page_h=1080, cols=None, margin=2.0,
                      keep_temp=False):
    """
    按页码分组并拼图。
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    temp_dir = out / "_temp_pages"

    for page_idx in range(max_pages):
        print(f"\n[页码 {page_idx+1}/{max_pages}] 收集页面...")

        # 收集该页码下所有 PDF 贡献的图片
        group_pngs = []
        for pdf_idx, pages in enumerate(all_pages):
            if page_idx < len(pages):
                temp_path = temp_dir / f"pdf{pdf_idx:02d}_pg{page_idx:03d}.png"
                temp_path.parent.mkdir(parents=True, exist_ok=True)
                pages[page_idx].save(temp_path)
                group_pngs.append(str(temp_path))
                print(f"    PDF[{pdf_idx}] 第{page_idx+1}页 → {temp_path.name}")
            else:
                print(f"    PDF[{pdf_idx}] 没有第{page_idx+1}页，跳过")

        if not group_pngs:
            print(f"    跳过（无页面）")
            continue

        # 拼图
        output_png = out / f"page_{page_idx:03d}.png"
        print(f"    拼合 {len(group_pngs)} 张图 → {output_png.name}")
        collage_png_pages_to_single(
            input_png=group_pngs,
            output_png=str(output_png),
            page_width=page_w,
            page_height=page_h,
            cols=cols,
            margin=margin,
        )
        print(f"    ✓ 已保存: {output_png}")

        # 清理临时文件
        if not keep_temp:
            for p in group_pngs:
                Path(p).unlink()

    # 清理空临时目录
    if not keep_temp and temp_dir.exists():
        try:
            temp_dir.rmdir()
        except OSError:
            pass  # 目录非空 — 用户设置了 keep_temp

    print(f"\n✓ 完成！共生成 {max_pages} 张拼图，保存在: {out.resolve()}")


def get_pdf_names(pdf_paths):
    """生成一个文件名映射说明，方便确认顺序。"""
    lines = []
    for i, p in enumerate(pdf_paths):
        lines.append(f"  PDF[{i}]: {Path(p).name}")
    return "\n".join(lines)

### 执行

In [23]:


# ─────────── 修改这里 ───────────
IO = {
    '1': {
        "PDF_PATHS": [
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_0_71/raw_yields_00.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_0_71/raw_yields_07.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_0_71/raw_yields_14.pdf",
        ],
        "OUTPUT_DIR": "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_0_71"
    },
    '2': {
        "PDF_PATHS": [
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_74_76/raw_yields_00.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_74_76/raw_yields_07.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_74_76/raw_yields_14.pdf",
        ],
        "OUTPUT_DIR": "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_74_76"
    },
    '3': {
        "PDF_PATHS": [
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_0_71/raw_yields_00.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_0_71/raw_yields_07.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_0_71/raw_yields_14.pdf",
        ],
        "OUTPUT_DIR": "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_0_71"
    },
    '4': {
        "PDF_PATHS": [
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76/raw_yields_00.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76/raw_yields_07.pdf",
            "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76/raw_yields_14.pdf",
        ],
        "OUTPUT_DIR": "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76"
    },
}

# OUTPUT_DIR = "/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76"   # 输出目录

# 拼图参数
DPI       = 800      # 从 PDF 提取页面的分辨率
PAGE_W    = 3600     # 拼图画布宽（像素）
PAGE_H    = 1000    # 拼图画布高（像素）
COLS      = 3     # 拼图列数；None = 自动按 16:9 估算
MARGIN    = 1.0      # 每个小图四周留白（像素）
KEEP_TEMP = False    # 是否保留中间临时 PNG（用于调试）
# ────────────────────────────────

for key, params in IO.items():
    print(f"\n\n{'#' * 60}\n处理配置 {key}\n{'#' * 60}")
    PDF_PATHS = params["PDF_PATHS"]
    OUTPUT_DIR = params["OUTPUT_DIR"]

    print("=" * 60)
    print("输入 PDF 顺序:")
    print(get_pdf_names(PDF_PATHS))
    print()

    print("步骤 1/2: 提取所有页面...")
    all_pages, max_pages = extract_all_pages(PDF_PATHS, dpi=DPI)
    print(f"\n共 {len(all_pages)} 个 PDF，最大页数: {max_pages}")

    print("\n" + "=" * 60)
    print("步骤 2/2: 按页码分组拼图...")
    group_and_collage(
        all_pages, max_pages,
        output_dir=OUTPUT_DIR,
        page_w=PAGE_W,
        page_h=PAGE_H,
        cols=COLS,
        margin=MARGIN,
        keep_temp=KEEP_TEMP,
    )



############################################################
处理配置 1
############################################################
输入 PDF 顺序:
  PDF[0]: raw_yields_00.pdf
  PDF[1]: raw_yields_07.pdf
  PDF[2]: raw_yields_14.pdf

步骤 1/2: 提取所有页面...
  正在读取: raw_yields_00.pdf
    → 2 页
  正在读取: raw_yields_07.pdf
    → 2 页
  正在读取: raw_yields_14.pdf
    → 2 页

共 3 个 PDF，最大页数: 2

步骤 2/2: 按页码分组拼图...

[页码 1/2] 收集页面...
    PDF[0] 第1页 → pdf00_pg000.png
    PDF[1] 第1页 → pdf01_pg000.png
    PDF[2] 第1页 → pdf02_pg000.png
    拼合 3 张图 → page_000.png
    ✓ 已保存: /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_0_71/page_000.png

[页码 2/2] 收集页面...
    PDF[0] 第2页 → pdf00_pg001.png
    PDF[1] 第2页 → pdf01_pg001.png
    PDF[2] 第2页 → pdf02_pg001.png
    拼合 3 张图 → page_001.png
    ✓ 已保存: /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/a_side/pt_0_71/page_001.png

✓ 完成！共生成 2 张拼图，保存在: /home/wuct/MetaData/DATA/PbPb/2

### 输出预览

列出生成的 PNG 文件及大小（方便确认是否正常）。

In [24]:
from pathlib import Path
out = Path(OUTPUT_DIR)
png_files = sorted(out.glob("page_*.png"))
print(f"输出目录: {out.resolve()}")
print(f"共生成 {len(png_files)} 个文件:\n")
for f in png_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")

输出目录: /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/raw_yields/b_side/pt_74_76
共生成 2 个文件:

  page_000.png  (436.9 KB)
  page_001.png  (412.7 KB)


### 附注

- PDF 中每个页面都用 PyMuPDF (fitz) 渲染成 RGB 位图，DPI 控制清晰度。
- `COLS = None` 时自动按 16:9 估算网格列数；也可以手动设 `COLS = 2` 等。
- 如果某个 PDF 页数比其他少，缺失的页码自动跳过（不会报错）。
- 设置 `KEEP_TEMP = True` 可以在 `_temp_pages/` 下保留中间提取的单页 PNG。